# tensor-unbind composite — cx13: unbind O,D from rays of shape (NR, 2, 3) and evaluate ray(t)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `tensor-unbind`, `ray-parametric-form`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "tensor-unbind"
DD_ATOM_IDS = ["tensor-unbind", "ray-parametric-form"]
DD_SUBTOPICS = ["Numpy: Indexing and selection", "Geometry: Ray parametric form"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

In ARENA part 1, the canonical ray layout is `rays: (NR, 2, 3)` — each ray is a stacked `(origin, direction)` pair along axis 1. To evaluate the ray's parametric form `P(t) = O + t * D` we first need to *separate* the origin and direction tensors. `tensor-unbind` along axis 1 does this in one named call, returning a tuple `(O, D)` where each has shape `(NR, 3)`.

The composition is `unbind` → `ray-parametric-form`. `unbind(rays, dim=1)` splits along the stacked-pair axis (no copy — each output is a stride view), and the parametric eval is just `O + t * D` with `t` broadcasting against the per-ray axis. The unbind is load-bearing: a wrong `dim` argument (e.g. `dim=0`) would give you `NR` tensors of shape `(2, 3)` instead of two tensors of shape `(NR, 3)`, and the parametric eval would silently give nonsense.

### Composite Exercise — unbind O,D from rays of shape (NR, 2, 3) and evaluate ray(t)

**Atoms exercised together**: `tensor-unbind`, `ray-parametric-form`

Implement `cx13_ray_at_t(rays, t_scalar)` that takes a ray batch of shape `(NR, 2, 3)` (axis 1 is the stacked `(origin, direction)` pair) and a scalar `t_scalar: float`, and returns the per-ray point `P(t) = O + t_scalar * D` of shape `(NR, 3)`.

1. **Unbind** along axis 1 to separate origin and direction: `O, D = t.unbind(rays, dim=1)`. Both `O` and `D` have shape `(NR, 3)`.
2. **Apply the parametric form** `O + t_scalar * D`. Broadcasting against the scalar is automatic.

Return shape `(NR, 3)`. The test verifies `O` and `D` were taken from the correct axis (axis 1, not axis 0) and that the parametric eval matches a hand-computed reference.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx13_ray_at_t(rays, t_scalar):
    raise NotImplementedError

def _test_cx13():
    # Case A: hand-built rays — origins at the column index, directions all unit-x.
    # rays[i, 0] = origin_i = (i, 0, 0); rays[i, 1] = direction_i = (1, 0, 0).
    NR = 5
    origins = t.stack([t.tensor([float(i), 0.0, 0.0]) for i in range(NR)])  # (NR, 3)
    directions = t.stack([t.tensor([1.0, 0.0, 0.0]) for _ in range(NR)])    # (NR, 3)
    rays = t.stack([origins, directions], dim=1)  # (NR, 2, 3)
    assert tuple(rays.shape) == (NR, 2, 3)

    # At t=2: P_i = (i + 2, 0, 0).
    out = cx13_ray_at_t(rays, 2.0)
    assert tuple(out.shape) == (NR, 3), f'expected (NR,3), got {tuple(out.shape)}'
    expected = t.stack([t.tensor([float(i) + 2.0, 0.0, 0.0]) for i in range(NR)])
    assert t.allclose(out, expected), f'parametric eval wrong: {out}\nexpected: {expected}'

    # Case B: t=0 should return origins exactly.
    out0 = cx13_ray_at_t(rays, 0.0)
    assert t.allclose(out0, origins), 'at t=0 result must equal origins'

    # Case C: random rays — cross-check against manual indexing.
    rays2 = t.randn(7, 2, 3)
    out2 = cx13_ray_at_t(rays2, 1.5)
    ref = rays2[:, 0, :] + 1.5 * rays2[:, 1, :]
    assert t.allclose(out2, ref), 'unbind axis is wrong — did you use dim=1?'

    # Case D: negative t (ray going backward).
    out_neg = cx13_ray_at_t(rays2, -0.5)
    ref_neg = rays2[:, 0, :] - 0.5 * rays2[:, 1, :]
    assert t.allclose(out_neg, ref_neg)
    _dd_passed.add('cx13')

_test_cx13()

<details><summary>Show solution — cx13</summary>

```python
def cx13_ray_at_t(rays, t_scalar):
    # Atom A (tensor-unbind): split the stacked (origin, direction) pair along axis 1.
    O, D = t.unbind(rays, dim=1)
    # Atom B (ray-parametric-form): P(t) = O + t * D.
    return O + t_scalar * D
```

`t.unbind(rays, dim=1)` returns a tuple of length 2 (the size of axis 1), each of shape `(NR, 3)`. Tuple-unpacking via `O, D = ...` is idiomatic. Common bug: using `dim=0` gives you `NR` tensors of shape `(2, 3)` (one per ray) — wrong axis. Another bug: using `rays[:, 0]` / `rays[:, 1]` works too but is less explicit about WHY we're splitting; unbind documents the intent ("this axis enumerates the stacked components").
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx13'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx13',
        'subtopics': ["Numpy: Indexing and selection", "Geometry: Ray parametric form"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()